# Joke Rating Prediction with Matrix Completion

## Business use case

Personalized content feeds benefit from predicting which items a user is likely to enjoy before explicit feedback exists. The Jester dataset provides a useful stress test because users score jokes on a wide range from -10 to 10 and the rating matrix is sparse.

## Objective

This notebook builds a user-joke matrix, hides a subset of observed ratings, and applies SoftImpute collaborative filtering. The model is evaluated against an average-rating baseline with in-sample and out-of-sample R².

## Result in context

The stored run reports **0.3463 out-of-sample R²** and **0.5143 in-sample R²**, the strongest validation R² among the three matrix-completion notebooks in this portfolio set.


## Step 1 — Load the Jester ratings

The ratings are read as user ID, joke ID, and score triplets. These records are the observed entries from which the sparse user-item matrix is constructed.


In [1]:
###import necessary libraries
!pip install fancyimpute
import sys
sys.path.append('/collaborativeFiltering/')
import os
os.chdir("/collaborativeFiltering/")
import numpy as np
import pandas as pd
from fancyimpute import BiScaler
from soft_impute import SoftImpute
from functionsCF import GenerateTrainingSet

In [2]:
rating = pd.read_csv('/collaborativeFiltering/jester-data-3.csv',sep=',').values
print(rating[:5, :])

[[ 1.    1.   -7.82]
 [ 1.    2.    8.79]
 [ 1.    3.   -9.66]
 [ 1.    4.   -8.16]
 [ 1.    5.   -7.52]]


## Step 2 — Build a compact user-joke matrix

Joke IDs are remapped to contiguous columns and known scores are inserted into an otherwise missing matrix. This representation allows low-rank methods to operate directly on the sparse preference surface.


In [3]:
matrix_incomplete = np.zeros((len(np.unique(rating[:,0])),len(np.unique(rating[:,1]))))

In [4]:
usedID = np.unique(rating[:,1])
for i in range(len(rating[:,1])):
    rating[:,1][i] = np.where(usedID==rating[:,1][i])[0][0] + 1

In [5]:
matrix_incomplete[:] = np.nan
indices = np.array(rating[:,0]-1).astype(int), np.array(rating[:,1]-1).astype(int)
matrix_incomplete[indices] = rating[:,2]

## Step 3 — Hold out observed ratings

Known entries are split into training and validation sets. The model only sees the training indices, which makes the validation error a real missing-rating prediction test.


In [6]:
train_indices, validation_indices = GenerateTrainingSet(rating[:,0],rating[:,1],0.8)
matrix_train = matrix_incomplete.copy()
matrix_train[:] = np.nan
matrix_train[train_indices] = matrix_incomplete[train_indices]

## Step 4 — Prepare the matrix for completion

The training matrix is passed through `BiScaler`. The fitted transformation is retained so completed scores can be converted back to the original Jester rating scale.


In [7]:
biscaler = BiScaler(scale_rows=False, scale_columns = False, max_iters = 50, verbose = False)
matrix_train_normalized = biscaler.fit_transform(matrix_train)

## Step 5 — Fit a nine-component SoftImpute model

SoftImpute learns a low-rank representation and fills the missing entries. Nine latent components define the capacity of the preference model in this run.


In [8]:
softImpute = SoftImpute(J = 9, maxit = 200, random_seed = 2022, verbose = False)

In [9]:
# Run the softImpute model on the normalized training set
# Call the output matrix_train_softImpute
matrix_train_softImpute=softImpute.fit(matrix_train_normalized)
matrix_train_filled_normalized=matrix_train_softImpute.predict(matrix_train_normalized,copyto=False)
matrix_train_filled=biscaler.inverse_transform(matrix_train_filled_normalized)

## Step 6 — Evaluate against the mean-rating baseline

The global training average supplies a simple reference predictor. Relative MSE reduction is reported as R² for both training and validation entries.


In [10]:
train_average = np.average(matrix_train[train_indices])

In [11]:
validation_mse= ((matrix_train_filled[validation_indices]-matrix_incomplete[validation_indices])**2).mean()
training_mse=((matrix_train_filled[train_indices]-matrix_incomplete[train_indices])**2).mean()
validation_mse_baseline=((train_average - matrix_incomplete[validation_indices]) ** 2).mean()
training_mse_baseline=((train_average - matrix_incomplete[train_indices])**2).mean()
print("out-of-sample R2: %.4f, in-sample R2: %.4f." % (1 - validation_mse / validation_mse_baseline, 1 - training_mse / training_mse_baseline))

out-of-sample R2: 0.3463, in-sample R2: 0.5143.


## Step 7 — Inspect a completed rating

A specific user-joke prediction is printed to verify the completed matrix produces interpretable scores on the original scale.


In [12]:
print("After matrix completion =", matrix_train_filled[100, 16])

After matrix completion = 1.8619063713464568


## Technical conclusions

The stored model reaches **0.3463 out-of-sample R²** and **0.5143 in-sample R²**. The smaller train-validation gap relative to the MovieLens experiment suggests better generalization in this particular setup, although most variation in individual ratings remains unexplained.

The next technical step would be systematic tuning of latent rank and regularization, followed by ranking-based evaluation. A recommender is ultimately judged by which items it surfaces, not only by squared error on ratings.

## Business conclusions

The experiment shows that shared taste patterns can support personalization even for subjective content such as humor. That makes the model useful as a baseline for feed ordering or candidate generation, especially when direct content features are weak.

## Limitations and next steps

The notebook does not address new users or new jokes. A hybrid system using metadata, exploration, and popularity priors would be more robust in a live product.
